In [17]:
import pandas as pd
import numpy as np

def extract_and_filter_ibtracs(file_path):
    print("1. Loading IBTrACS Western Pacific Dataset...")
    df = pd.read_csv(file_path, skiprows=[1], low_memory=False, na_values=[' ', ''])
    
    print("2. Pruning Columns...")
    core_columns = [
        'SID', 'SEASON', 'ISO_TIME', 'NATURE', 'TRACK_TYPE',
        'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES',
        'USA_WIND', 'USA_PRES', 'DIST2LAND'
    ]
    df = df[core_columns].copy()
    
    print("3. Casting Data Types...")
    df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'])
    numeric_cols = ['TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES', 
                    'USA_WIND', 'USA_PRES', 'DIST2LAND']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    print("4. Applying Meteorological Filters...")
    df = df[df['TRACK_TYPE'] == 'main']
    df = df[~df['NATURE'].isin(['ET', 'DS'])]
    
    print("5. Applying the 'DIST2LAND' Geospatial Shortcut...")
    df['is_over_land'] = df['DIST2LAND'] == 0
    
    print("6. Executing 6-Hour Temporal Resampling...")
    df = df.sort_values(['SID', 'ISO_TIME'])
    
    def resample_storm(storm_df):
        sid = storm_df['SID'].iloc[0]
        storm_df = storm_df.set_index('ISO_TIME')
        storm_df = storm_df.resample('6h').asfreq()
        storm_df.index.name = 'ISO_TIME'
        storm_df['SID'] = sid
        return storm_df
    
    df_6h = df.groupby('SID', group_keys=False).apply(resample_storm)
    df_6h = df_6h.reset_index()
    
    print("Data extraction complete.")
    return df_6h

In [18]:
# --- Execution ---
file_name = 'ibtracs.WP.list.v04r01.csv' 

# This line actually triggers the engine from Cell 1
processed_df = extract_and_filter_ibtracs(file_name)

# Display the results
print(f"\nFinal Shape: {processed_df.shape[0]} rows, {processed_df.shape[1]} columns")
display(processed_df.head(10))

1. Loading IBTrACS Western Pacific Dataset...
2. Pruning Columns...
3. Casting Data Types...
4. Applying Meteorological Filters...
5. Applying the 'DIST2LAND' Geospatial Shortcut...
6. Executing 6-Hour Temporal Resampling...
Data extraction complete.

Final Shape: 106662 rows, 13 columns


C:\Users\user\AppData\Local\Temp\ipykernel_27116\1267082598.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_6h = df.groupby('SID', group_keys=False).apply(resample_storm)


,ISO_TIME,SID,SEASON,NATURE,TRACK_TYPE,TOKYO_LAT,TOKYO_LON,TOKYO_WIND,TOKYO_PRES,USA_WIND,USA_PRES,DIST2LAND,is_over_land
0,1884-06-24 12:00:00,1884177N17124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1884-06-24 18:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,145.0,False
2,1884-06-25 00:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,77.0,False
3,1884-06-25 06:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,10.0,False
4,1884-06-25 12:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
5,1884-06-25 18:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
6,1884-06-26 00:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
7,1884-06-26 06:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,31.0,False
8,1884-06-26 12:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,90.0,False
9,1884-06-26 18:00:00,1884177N17124,1884.0,NR,main,NaN,NaN,NaN,NaN,NaN,NaN,151.0,False


In [19]:
print("1. Slicing to the Modern Satellite Era (1980+)...")
# Drop pre-1980 storms where data fidelity is too low for ML
df_modern = processed_df[processed_df['SEASON'] >= 1980].copy()

print("2. Interpolating Small Trajectory Gaps...")
cols_to_interpolate = ['TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES']

# Using .transform() instead of .apply() ensures we don't lose the SID column
df_modern[cols_to_interpolate] = df_modern.groupby('SID')[cols_to_interpolate].transform(
    lambda x: x.interpolate(method='linear', limit=2, limit_direction='forward')
)

# Drop any rows that STILL have missing coordinates after interpolation
# (Meaning the storm dropped off radar for more than 12 hours)
df_clean = df_modern.dropna(subset=['TOKYO_LAT', 'TOKYO_LON']).copy()

print(f"Modern Era Shape: {df_clean.shape[0]} rows.")
display(df_clean[['SID', 'ISO_TIME', 'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES']].head(10))

1. Slicing to the Modern Satellite Era (1980+)...
2. Interpolating Small Trajectory Gaps...
Modern Era Shape: 35412 rows.


,SID,ISO_TIME,TOKYO_LAT,TOKYO_LON,TOKYO_WIND,TOKYO_PRES
61125,1980094N02179,1980-04-05 00:00:00,9.5,180.0,NaN,1002.0
61126,1980094N02179,1980-04-05 06:00:00,10.4,178.6,NaN,1000.0
61127,1980094N02179,1980-04-05 12:00:00,11.8,177.7,40.0,994.0
61128,1980094N02179,1980-04-05 18:00:00,13.4,177.3,45.0,992.0
61129,1980094N02179,1980-04-06 00:00:00,14.8,177.2,45.0,992.0
61130,1980094N02179,1980-04-06 06:00:00,15.8,177.2,45.0,990.0
61131,1980094N02179,1980-04-06 12:00:00,16.7,177.6,60.0,985.0
61132,1980094N02179,1980-04-06 18:00:00,17.6,178.1,55.0,985.0
61133,1980094N02179,1980-04-07 00:00:00,18.5,178.9,45.0,990.0
61134,1980094N02179,1980-04-07 06:00:00,19.5,179.8,45.0,992.0


In [20]:
print("1. Preparing Physical Wind-Pressure Math....")
# Create a strict mask: Only calculate physics for storms over the ocean
mask_ocean = df_clean['is_over_land'] == False

print("2. Imputing Missing Pressure from Wind...")
# Equation: P_c = 1010 - (V_1min / 6.7) ** 1.5528
mask_missing_pres = df_clean['TOKYO_PRES'].isna() & df_clean['TOKYO_WIND'].notna() & mask_ocean
# Convert JMA 10-min wind to 1-min equivalent first
v_1min = df_clean.loc[mask_missing_pres, 'TOKYO_WIND'] * 1.14
df_clean.loc[mask_missing_pres, 'TOKYO_PRES'] = 1010 - (v_1min / 6.7) ** (1 / 0.644)

print("3. Imputing Missing Wind from Pressure...")
# Equation: V_1min = 6.7 * (1010 - P_c) ** 0.644
mask_missing_wind = df_clean['TOKYO_WIND'].isna() & df_clean['TOKYO_PRES'].notna() & mask_ocean

# Sub-mask A: Storms strong enough to have a pressure drop (P < 1010)
mask_calc_wind = mask_missing_wind & (df_clean['TOKYO_PRES'] < 1010)
v_1min_calc = 6.7 * ((1010 - df_clean.loc[mask_calc_wind, 'TOKYO_PRES']) ** 0.644)
# Convert 1-min back to JMA 10-min standard to maintain dataset consistency
df_clean.loc[mask_calc_wind, 'TOKYO_WIND'] = v_1min_calc / 1.14

# Sub-mask B: Weak baseline depressions (P >= 1010)
mask_base_wind = mask_missing_wind & (df_clean['TOKYO_PRES'] >= 1010)
df_clean.loc[mask_base_wind, 'TOKYO_WIND'] = 15.0 # Set to baseline tropical disturbance wind (15 knots)

print("4. Dropping Unrecoverable Rows...")
# Drop any rows where BOTH wind and pressure were missing, or it was missing over land
df_clean = df_clean.dropna(subset=['TOKYO_WIND', 'TOKYO_PRES']).copy()

print(f"Post-Imputation Shape: {df_clean.shape[0]} rows.")
display(df_clean[['SID', 'ISO_TIME', 'TOKYO_WIND', 'TOKYO_PRES', 'is_over_land']].head(10))

1. Preparing Physical Wind-Pressure Math....
2. Imputing Missing Pressure from Wind...
3. Imputing Missing Wind from Pressure...
4. Dropping Unrecoverable Rows...
Post-Imputation Shape: 34391 rows.


,SID,ISO_TIME,TOKYO_WIND,TOKYO_PRES,is_over_land
61125,1980094N02179,1980-04-05 00:00:00,22.426418,1002.0,False
61126,1980094N02179,1980-04-05 06:00:00,25.892260,1000.0,False
61127,1980094N02179,1980-04-05 12:00:00,40.000000,994.0,False
61128,1980094N02179,1980-04-05 18:00:00,45.000000,992.0,False
61129,1980094N02179,1980-04-06 00:00:00,45.000000,992.0,False
61130,1980094N02179,1980-04-06 06:00:00,45.000000,990.0,False
61131,1980094N02179,1980-04-06 12:00:00,60.000000,985.0,False
61132,1980094N02179,1980-04-06 18:00:00,55.000000,985.0,False
61133,1980094N02179,1980-04-07 00:00:00,45.000000,990.0,False
61134,1980094N02179,1980-04-07 06:00:00,45.000000,992.0,False


In [21]:
print("1. Calculating Seasonal Physics (Sine/Cosine)...")
# Extract day of the year and convert to a cyclical wave
# This tells the AI the difference between a July storm and a December storm
df_clean['day_of_year'] = df_clean['ISO_TIME'].dt.dayofyear
df_clean['sin_season'] = np.sin(2 * np.pi * df_clean['day_of_year'] / 365.25)
df_clean['cos_season'] = np.cos(2 * np.pi * df_clean['day_of_year'] / 365.25)

print("2. Calculating Velocity Differentials...")
# Ensure chronological order
df_clean = df_clean.sort_values(by=['SID', 'ISO_TIME'])

# How much did the storm move in the last 6 hours? (Momentum)
df_clean['delta_lat'] = df_clean.groupby('SID')['TOKYO_LAT'].diff()
df_clean['delta_lon'] = df_clean.groupby('SID')['TOKYO_LON'].diff()

print("3. Building the History Sliding Window (t-6 and t-12)...")
cols_to_shift = ['TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES']

for col in cols_to_shift:
    # .shift(1) pulls the value from 6 hours ago
    df_clean[f'{col}_t-6'] = df_clean.groupby('SID')[col].shift(1)
    # .shift(2) pulls the value from 12 hours ago
    df_clean[f'{col}_t-12'] = df_clean.groupby('SID')[col].shift(2)

print("4. Formulating the Predictive Targets (t+6)...")
# .shift(-1) looks into the FUTURE to get the "Answer" the AI must guess
df_clean['TARGET_LAT'] = df_clean.groupby('SID')['TOKYO_LAT'].shift(-1)
df_clean['TARGET_LON'] = df_clean.groupby('SID')['TOKYO_LON'].shift(-1)

print("5. Dropping Incomplete Windows...")
# Shifting creates NaNs (e.g., the first 12 hours of a storm have no history, 
# and the last 6 hours have no future target). We must drop these incomplete rows.
df_ml = df_clean.dropna().copy()

print(f"Final Machine Learning Dataset Shape: {df_ml.shape[0]} rows.")
display(df_ml[['SID', 'ISO_TIME', 'TOKYO_LAT_t-12', 'TOKYO_LAT_t-6', 'TOKYO_LAT', 'TARGET_LAT']].head(5))

1. Calculating Seasonal Physics (Sine/Cosine)...
2. Calculating Velocity Differentials...
3. Building the History Sliding Window (t-6 and t-12)...
4. Formulating the Predictive Targets (t+6)...
5. Dropping Incomplete Windows...
Final Machine Learning Dataset Shape: 14476 rows.


,SID,ISO_TIME,TOKYO_LAT_t-12,TOKYO_LAT_t-6,TOKYO_LAT,TARGET_LAT
86452,2001125N05129,2001-05-10 00:00:00,11.8,12.3,13.1,14.1
86453,2001125N05129,2001-05-10 06:00:00,12.3,13.1,14.1,15.3
86454,2001125N05129,2001-05-10 12:00:00,13.1,14.1,15.3,16.2
86455,2001125N05129,2001-05-10 18:00:00,14.1,15.3,16.2,17.1
86456,2001125N05129,2001-05-11 00:00:00,15.3,16.2,17.1,17.6


In [22]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
import numpy as np

print("1. Defining Features and Targets...")
# The inputs (Questions)
features = [
    'TOKYO_LAT_t-12', 'TOKYO_LON_t-12', 'TOKYO_WIND_t-12', 'TOKYO_PRES_t-12',
    'TOKYO_LAT_t-6', 'TOKYO_LON_t-6', 'TOKYO_WIND_t-6', 'TOKYO_PRES_t-6',
    'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES',
    'delta_lat', 'delta_lon', 'sin_season', 'cos_season', 'is_over_land'
]

# The outputs (Answers)
targets = ['TARGET_LAT', 'TARGET_LON']

print("2. Executing Chronological Split (1980-2018 vs 2019+)...")
df_ml['year'] = df_ml['ISO_TIME'].dt.year

# Training Set (The textbooks)
train_mask = df_ml['year'] <= 2018
X_train = df_ml.loc[train_mask, features]
y_train = df_ml.loc[train_mask, targets]

# Testing Set (The final exam)
test_mask = df_ml['year'] >= 2019
X_test = df_ml.loc[test_mask, features]
y_test = df_ml.loc[test_mask, targets]

print(f"   -> Training on {X_train.shape[0]} storm observations.")
print(f"   -> Testing on {X_test.shape[0]} holdout observations.")

print("3. Initializing XGBoost Engine...")
# We start with robust, standard hyperparameters
base_xgb = xgb.XGBRegressor(
    n_estimators=100,      # Number of decision trees
    learning_rate=0.1,     # How aggressively it corrects errors
    max_depth=6,           # How complex each tree can be
    random_state=42,       # Ensures reproducibility for your pitch
    n_jobs=-1              # Uses all your CPU cores for speed
)

# Wrap it to predict both Lat and Lon at the same time
model = MultiOutputRegressor(base_xgb)

print("4. Training the Model (This may take a few seconds)...")
model.fit(X_train, y_train)

print("5. Generating Predictions for the Test Set...")
predictions = model.predict(X_test)

print("Training Complete! The model is ready for spatial evaluation.")

1. Defining Features and Targets...
2. Executing Chronological Split (1980-2018 vs 2019+)...
   -> Training on 11048 storm observations.
   -> Testing on 3428 holdout observations.
3. Initializing XGBoost Engine...
4. Training the Model (This may take a few seconds)...


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:is_over_land: object

In [23]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
import numpy as np

print("1. Defining Features and Targets...")
# The inputs (Questions)
features = [
    'TOKYO_LAT_t-12', 'TOKYO_LON_t-12', 'TOKYO_WIND_t-12', 'TOKYO_PRES_t-12',
    'TOKYO_LAT_t-6', 'TOKYO_LON_t-6', 'TOKYO_WIND_t-6', 'TOKYO_PRES_t-6',
    'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_WIND', 'TOKYO_PRES',
    'delta_lat', 'delta_lon', 'sin_season', 'cos_season', 'is_over_land'
]

# The outputs (Answers)
targets = ['TARGET_LAT', 'TARGET_LON']

# --- THE FIX: Force the object column into strict integers (1 or 0) ---
df_ml['is_over_land'] = df_ml['is_over_land'].astype(int)
# Force all other feature columns to float just to be absolutely safe for XGBoost
for col in features:
    df_ml[col] = df_ml[col].astype(float)
# ---------------------------------------------------------------------

print("2. Executing Chronological Split (1980-2018 vs 2019+)...")
df_ml['year'] = df_ml['ISO_TIME'].dt.year

# Training Set (The textbooks)
train_mask = df_ml['year'] <= 2018
X_train = df_ml.loc[train_mask, features]
y_train = df_ml.loc[train_mask, targets]

# Testing Set (The final exam)
test_mask = df_ml['year'] >= 2019
X_test = df_ml.loc[test_mask, features]
y_test = df_ml.loc[test_mask, targets]

print(f"   -> Training on {X_train.shape[0]} storm observations.")
print(f"   -> Testing on {X_test.shape[0]} holdout observations.")

print("3. Initializing XGBoost Engine...")
# We start with robust, standard hyperparameters
base_xgb = xgb.XGBRegressor(
    n_estimators=100,      # Number of decision trees
    learning_rate=0.1,     # How aggressively it corrects errors
    max_depth=6,           # How complex each tree can be
    random_state=42,       # Ensures reproducibility for your pitch
    n_jobs=-1              # Uses all your CPU cores for speed
)

# Wrap it to predict both Lat and Lon at the same time
model = MultiOutputRegressor(base_xgb)

print("4. Training the Model (This may take a few seconds)...")
model.fit(X_train, y_train)

print("5. Generating Predictions for the Test Set...")
predictions = model.predict(X_test)

print("Training Complete! The model is ready for spatial evaluation.")

1. Defining Features and Targets...
2. Executing Chronological Split (1980-2018 vs 2019+)...
   -> Training on 11048 storm observations.
   -> Testing on 3428 holdout observations.
3. Initializing XGBoost Engine...
4. Training the Model (This may take a few seconds)...
5. Generating Predictions for the Test Set...
Training Complete! The model is ready for spatial evaluation.


In [24]:
import numpy as np
import pandas as pd

print("1. Defining the Haversine Distance Formula...")
def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in kilometers
    R = 6371.0
    
    # Convert degrees to radians for spherical mathematics
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return R * c

print("2. Calculating Cross-Track Error for the 2019+ Test Set...")
# Extract actual historical coordinates
actual_lat = y_test['TARGET_LAT'].values
actual_lon = y_test['TARGET_LON'].values

# Extract XGBoost predicted coordinates (Column 0 is Lat, Column 1 is Lon)
pred_lat = predictions[:, 0]
pred_lon = predictions[:, 1]

# Calculate the physical distance in kilometers for all 3,428 predictions
errors_km = haversine(actual_lat, actual_lon, pred_lat, pred_lon)

print("\n3. Final Model Accuracy Metrics (6-Hour Forecast Horizon):")
mean_error = np.mean(errors_km)
median_error = np.median(errors_km)

print(f"   -> Mean Error:   {mean_error:.2f} km")
print(f"   -> Median Error: {median_error:.2f} km")

print("\n4. Sample: AI Prediction vs. Reality")
# Construct a DataFrame to visually compare the results
results_df = pd.DataFrame({
    'Actual_Lat': np.round(actual_lat, 2),
    'Pred_Lat': np.round(pred_lat, 2),
    'Actual_Lon': np.round(actual_lon, 2),
    'Pred_Lon': np.round(pred_lon, 2),
    'Error_km': np.round(errors_km, 2)
})

display(results_df.head(10))

1. Defining the Haversine Distance Formula...
2. Calculating Cross-Track Error for the 2019+ Test Set...

3. Final Model Accuracy Metrics (6-Hour Forecast Horizon):
   -> Mean Error:   49.91 km
   -> Median Error: 40.15 km

4. Sample: AI Prediction vs. Reality


,Actual_Lat,Pred_Lat,Actual_Lon,Pred_Lon,Error_km
0,6.3,6.47,110.2,110.419998,30.31
1,6.0,6.19,109.9,109.379997,61.26
2,5.8,5.55,109.5,109.739998,38.54
3,5.9,5.53,108.6,109.050003,64.65
4,6.2,5.48,108.0,107.849998,81.26
5,6.3,6.35,107.0,107.260002,29.28
6,6.0,5.80,105.8,106.349998,65.11
7,5.8,5.20,105.0,104.379997,95.88
8,6.1,5.28,104.1,104.949997,130.42
9,6.8,5.79,103.4,103.059998,118.34


In [25]:
import requests
import json

print("1. Initializing SEABeacon API Client...")
# The local address where your FastAPI server is listening
API_URL = "http://localhost:8000/api/v1/spatial-conversion"

print("2. Extracting Live Prediction from XGBoost Engine...")
# We will simulate a live event by grabbing the very first storm from your 2019+ test set.
# We cast to float because FastAPI requires standard Python floats, not NumPy floats.
sample_lat = float(pred_lat[0])  
sample_lon = float(pred_lon[0])  


# Inject a simulated direct strike on Eastern Samar
#sample_lat = 14.0583
#sample_lon = 108.2772

# We will use your highly accurate median error metric for the uncertainty cone
error_radius = 40.15 

# Construct the JSON payload exactly matching your Pydantic schema
payload = {
    "latitude": sample_lat,
    "longitude": sample_lon,
    "cross_track_error_km": error_radius
}

print(f"   -> Target Locked: Lat {sample_lat:.2f}, Lon {sample_lon:.2f}")
print(f"   -> Applying {error_radius}km uncertainty radius...")

print("3. Firing POST Request to Spatial Backend...")
try:
    # Send the data to your FastAPI server
    response = requests.post(API_URL, json=payload)
    
    # Evaluate the server's response
    if response.status_code == 200:
        print("\n✅ ALERT GENERATED SUCCESSFULLY:")
        # Pretty-print the JSON response so you can see the PostGIS output
        print(json.dumps(response.json(), indent=4))
    else:
        print(f"\n❌ API Error {response.status_code}: {response.text}")

except requests.exceptions.ConnectionError:
    print("\n❌ Connection Refused. Ensure main.py is running in a separate terminal.")

1. Initializing SEABeacon API Client...
2. Extracting Live Prediction from XGBoost Engine...
   -> Target Locked: Lat 6.47, Lon 110.42
   -> Applying 40.15km uncertainty radius...
3. Firing POST Request to Spatial Backend...

✅ ALERT GENERATED SUCCESSFULLY:
{
    "alert_status": "NO_IMPACT_DETECTED",
    "impacted_count": 0,
    "impacted_regions": []
}


In [29]:
import requests
import json
import numpy as np

print("1. Initializing Multi-Step Trajectory Engine...")
API_URL = "http://localhost:8000/api/v1/spatial-conversion"

MEDIAN_ERROR_6H = 40.15  

forecast_intervals = {
    1: "6-Hour Forecast",
    2: "12-Hour Forecast",
    4: "24-Hour Forecast",
    8: "48-Hour Forecast",
    12: "72-Hour Forecast"
}
max_steps = max(forecast_intervals.keys())

# --- SIMULATED INPUT INITIALIZATION ---
if 'pred_lat' in locals() or 'pred_lat' in globals():
    current_lat = float(pred_lat[0])
    current_lon = float(pred_lon[0])
else:
    current_lat = 6.47
    current_lon = 110.42

print(f"Initial Storm Position Secured: Lat {current_lat:.2f}, Lon {current_lon:.2f}\n")
print("2. Launching Iterative Forecasting Loop...")

trajectory_log = {}

for step in range(1, max_steps + 1):
    
    simulated_delta_lat = 0.15  
    simulated_delta_lon = -0.45 
    
    current_lat += simulated_delta_lat
    current_lon += simulated_delta_lon
    
    compounded_error = MEDIAN_ERROR_6H * np.sqrt(step)
    
    if step in forecast_intervals:
        horizon_name = forecast_intervals[step]
        time_elapsed = step * 6
        
        print(f"--- Executing API Pipeline for {horizon_name} (+{time_elapsed}hrs) ---")
        
        payload = {
            "latitude": round(current_lat, 4),
            "longitude": round(current_lon, 4),
            "cross_track_error_km": round(compounded_error, 2)
        }
        
        try:
            response = requests.post(API_URL, json=payload)
            
            if response.status_code == 200:
                data = response.json()
                print(f"    Target Coordinate: Lat {payload['latitude']}, Lon {payload['longitude']}")
                print(f"    Expanded Radius: {payload['cross_track_error_km']} km")
                
                # Format the impact text dynamically based on results
                if data['impacted_count'] > 0:
                    # Construct a clear string combining Country and Province names
                    provinces_list = [f"{region['country']} ({region['province']})" for region in data['impacted_regions']]
                    impact_details = ", ".join(provinces_list)
                    print(f"    Status: {data['alert_status']} | Impacted Regions: {impact_details}")
                else:
                    print(f"    Status: {data['alert_status']} | Impacted Regions: None")
                
                trajectory_log[f"{time_elapsed}h"] = {
                    "coordinates": [payload["latitude"], payload["longitude"]],
                    "error_radius_km": payload["cross_track_error_km"],
                    "impacted_areas": data["impacted_regions"]
                }
            else:
                print(f"    ❌ API Error {response.status_code}: {response.text}")
                
        except requests.exceptions.ConnectionError:
            print("    ❌ Connection Refused. Please run main.py in a separate terminal context.")
            break
        print("-" * 60)

print("\n3. Full 72-Hour Trajectory Analysis Complete.")

1. Initializing Multi-Step Trajectory Engine...
Initial Storm Position Secured: Lat 6.47, Lon 110.42

2. Launching Iterative Forecasting Loop...
--- Executing API Pipeline for 6-Hour Forecast (+6hrs) ---
    Target Coordinate: Lat 6.6154, Lon 109.968
    Expanded Radius: 40.15 km
    Status: NO_IMPACT_DETECTED | Impacted Regions: None
------------------------------------------------------------
--- Executing API Pipeline for 12-Hour Forecast (+12hrs) ---
    Target Coordinate: Lat 6.7654, Lon 109.518
    Expanded Radius: 56.78 km
    Status: NO_IMPACT_DETECTED | Impacted Regions: None
------------------------------------------------------------
--- Executing API Pipeline for 24-Hour Forecast (+24hrs) ---
    Target Coordinate: Lat 7.0654, Lon 108.618
    Expanded Radius: 80.3 km
    Status: NO_IMPACT_DETECTED | Impacted Regions: None
------------------------------------------------------------
--- Executing API Pipeline for 48-Hour Forecast (+48hrs) ---
    Target Coordinate: Lat 7.665